In [10]:
import os
import boto3
import requests

# aws s3 credentials
aws_access_key_id = os.getenv('aws_access_key_id')
aws_secret_access_key = os.getenv('aws_secret_access_key')




In [11]:
# Create aws session & s3 client

# S3 Location
S3_BUCKET = 'cm-aws-s3-data-source'
# S3_FOLDER_PATH = 'organization/sales'

# create a authorized session using boto3 + credentials
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)
# create s3 client to interact with AWS S3
s3 = session.client(service_name='s3')

SA Metro GTFS
- Loading Mode: Initial Full Load + Incremental Load
    - Initial Full Load: Load the previous 10 versions of GTFS
    - Incremental Load:
        - Get the latest version in `landing zone` & latest version available from API.
        - Execute load if the API's latest version has not been ingested.  
- Partitioned Destination: `<landing_bucket>/gtfs/<FEED VERSION>`

In [ ]:
# FULL LOAD
# 1. read data from SA Metro GTFS feed
# 2. upload feed version into landing bucket
base_url = "http://gtfs.adelaidemetro.com.au/v1"
latest_version_number = "static/latest/version.txt"
latest_version_feed = "static/latest/google_transit.zip"

destination_bucket = 'cm-aws-s3-destination'

# for chunk in res.iter_content(chunk_size=128):
# 1600 1610
os.makedirs('downloaded',exist_ok=True)
for version in list(range(1600,1611)):
    res = requests.get(f'{base_url}/static/{version}/google_transit.zip')

    if(res.ok):
        with open(f'downloaded/{version}_google_transit.zip','wb') as file:
            for chunk in res.iter_content(chunk_size=128):
                file.write(chunk)
            print(f'file {version} downloaded')
    else:
        print('bad request')



    

file 1600 downloaded
file 1601 downloaded
file 1602 downloaded
file 1603 downloaded
file 1604 downloaded
file 1605 downloaded
file 1606 downloaded
file 1607 downloaded
file 1608 downloaded
file 1609 downloaded
file 1610 downloaded


In [ ]:
# INCREMENTAL LOAD
# 1. Get: API's latest version & AWS S3's versions
# 2. Compare & logging whether the latest version is already ingested
# 3. Execute the ingestion if not already ingested

# Practice: define script into tasks / functions
# e.g., Func1: Get API Latest version
# Func2: Get AWS S3 versions and compare if API latest version is already ingested -> True/False
# Func3: Execute the API latest version ingestion if Func2 return False


def get_latest_version():
    request_headers={"Content-Type": "application/octet-stream"}
    res = requests.get(f'{base_url}/{latest_version_number}',headers=request_headers,stream=True)
    return res.text
